In [ ]:
# Wildfire Ignition Modelling Pipeline
# Load all raster datasets from the current working directory.
# This notebook reproduces the figures and evaluation reported in the manuscript.

import numpy as np
import rasterio
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

plt.style.use('default')

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f5f5f5",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": "#dddddd",
    "grid.linestyle": "-",
    "grid.linewidth": 0.8,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})


# =========================
# 1. LOAD DATA
# =========================
def load_raster(path):
    with rasterio.open(path) as src:
        return src.read(1)

ntl2 = load_raster("ntl2.tif") 
roads = load_raster("road_density.tif") 
fragmentation = load_raster("fragmentation.tif") 
fire = load_raster("fire_labels2.tif") 
ndvi = load_raster("NDVI_TN_KA.tif") 
lst = load_raster("LST_TN_KA.tif") 
builtup = load_raster("builtup_density_TN_KA.tif") 
aspect = load_raster("aspect_TN_KA.tif") 
# optional 
vpd = load_raster("VPD_TN_KA_FIXED.tif")
print("ntl:", ntl2.shape)
print("roads:", roads.shape)
print("frag:", fragmentation.shape)
print("ndvi:", ndvi.shape)
print("lst:", lst.shape)
print("builtup:", builtup.shape)
print("aspect:", aspect.shape)
print("vpd:", vpd.shape)

# =========================
# 2. FLATTEN
# =========================
ntl2_f = ntl2.flatten()
roads_f = roads.flatten()
frag_f = fragmentation.flatten()
fire_f = fire.flatten()
ndvi_f = ndvi.flatten()
lst_f = lst.flatten()
builtup_f = builtup.flatten()
aspect_f = aspect.flatten()
vpd_f = vpd.flatten()  # optional

# =========================
# 3. MASK
# =========================
mask = (
    ~np.isnan(ntl2_f) &
    ~np.isnan(roads_f) &
    ~np.isnan(frag_f) &
    ~np.isnan(ndvi_f) &
    ~np.isnan(lst_f) &
    ~np.isnan(builtup_f) &
    ~np.isnan(aspect_f)
    # add this only if using VPD
    & ~np.isnan(vpd_f)
)

ntl2_clean = ntl2_f[mask]
roads_clean = roads_f[mask]
frag_clean = frag_f[mask]
ndvi_clean = ndvi_f[mask]
y = fire_f[mask]
builtup_clean = builtup_f[mask]
aspect_clean = aspect_f[mask]
# optional
vpd_clean = vpd_f[mask]

# =========================
# 4. TRANSFORM + NORMALIZE
# =========================
ntl2_clean = np.log1p(ntl2_clean)

def normalize(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

ntl2_clean = normalize(ntl2_clean)
roads_clean = normalize(roads_clean)
frag_clean = normalize(frag_clean)
ndvi_clean = normalize(ndvi_clean)
lst_clean = normalize(lst_f[mask])
builtup_clean = builtup_f[mask]
aspect_clean = aspect_f[mask]
# optional
vpd_clean = vpd_f[mask]

# invert NDVI (important)
ndvi_clean = 1 - ndvi_clean

# =========================
# 5. FEATURE ENGINEERING
# =========================
interaction = ntl2_clean * frag_clean * (1 + roads_clean)
ntl_sq = ntl2_clean ** 2
frag_sq = frag_clean ** 2
risk_combo = ndvi_clean * frag_clean * roads_clean

# human pressure upgrade
human_pressure = ntl2_clean * (1 + roads_clean) * (1 + builtup_clean)

# terrain interaction
terrain_risk = aspect_clean * frag_clean

# optional climate interaction
vpd_ndvi = vpd_clean * ndvi_clean  # optional

#wui pi
human = (ntl2_clean + roads_clean + builtup_clean) / 3
fuel = (ndvi_clean + frag_clean) / 2

wui_pressure = human * fuel


X = np.column_stack([
    ntl2_clean,
    frag_clean,
    roads_clean,
    ndvi_clean,
    lst_clean,
    builtup_clean,
    aspect_clean,
    vpd_clean,         # NEW
    interaction,
    ntl_sq,
    frag_sq,
    risk_combo,
    human_pressure,
    terrain_risk,
    human,
    fuel,
    wui_pressure,
    vpd_ndvi
])

# =========================
# 6. SCALE
# =========================
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

# =========================
# 7. BALANCE DATA
# =========================
from sklearn.utils import resample

X_fire = X[y == 1]
X_no = X[y == 0]

X_no_down = resample(X_no, n_samples=len(X_fire)*10, random_state=42)

X_bal = np.vstack([X_fire, X_no_down])
y_bal = np.array([1]*len(X_fire) + [0]*len(X_no_down))

ntl2 = gaussian_filter(ntl2, sigma=2)
roads = gaussian_filter(roads, sigma=2)
fragmentation = gaussian_filter(fragmentation, sigma=2)
ndvi = gaussian_filter(ndvi, sigma=2)
lst = gaussian_filter(lst, sigma=2)

# =========================
# 8. TRAIN MODELS
# =========================

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# --- Random Forest (existing)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=20,
    class_weight={0:1, 1:20},
    n_jobs=-1,
    random_state=42
)
rf.fit(X_bal, y_bal)

# --- Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_bal, y_bal)

# --- XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=20,
    n_jobs=-1,
    random_state=42
)
xgb.fit(X_bal, y_bal)

# =========================
# 9. EVALUATION
# =========================
from sklearn.metrics import roc_auc_score

pred_rf = rf.predict_proba(X)[:,1]
pred_lr = lr.predict_proba(X)[:,1]
pred_xgb = xgb.predict_proba(X)[:,1]

print("RF AUC:", round(roc_auc_score(y, pred_rf), 3))
print("LR AUC:", round(roc_auc_score(y, pred_lr), 3))
print("XGB AUC:", round(roc_auc_score(y, pred_xgb), 3))

# =========================
# 10. BEST THRESHOLD + F1
# =========================

models = {
    "RF": rf,
    "LR": lr,
    "XGB": xgb
}

from sklearn.metrics import f1_score, classification_report
from sklearn.metrics import precision_recall_curve,auc
import numpy as np

h, w = ntl2.shape

left_mask = np.zeros_like(ntl2, dtype=bool)
left_mask[:, :w//2] = True
right_mask = ~left_mask

left_f = left_mask.flatten()[mask]
right_f = right_mask.flatten()[mask]

X_train = X[left_f]
y_train = y[left_f]

X_test = X[right_f]
y_test = y[right_f]

from sklearn.metrics import roc_curve, precision_recall_curve

# Figure 8: ROC Curves
plt.figure(figsize=(6,6))

for name, model in models.items():
    pred = model.predict_proba(X)[:,1]
    fpr, tpr, _ = roc_curve(y, pred)
    plt.plot(fpr, tpr, linewidth=2, label=name)

plt.plot([0,1], [0,1], linestyle='--', alpha=0.5)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")

plt.legend()

plt.tight_layout()
plt.savefig("roc_curves.png", dpi=300)
plt.show()


for name, model in models.items():
    print(f"\n===== {name} =====")

    pred = model.predict_proba(X)[:,1]

    print("AUC:", round(roc_auc_score(y, pred), 3))

     # ---------- PR CURVE ----------
    precision, recall, thresholds = precision_recall_curve(y, pred)

    # ---------- PR-AUC ----------
    pr_auc = auc(recall, precision)
    print("PR-AUC:", round(pr_auc, 3))

    # ---------- THRESHOLD ----------
    precision, recall, thresholds = precision_recall_curve(y, pred)
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt
    
    # Predictions using your chosen threshold
    y_pred = (pred_valid >= best_thresh).astype(int)
    
    cm = confusion_matrix(y_valid, y_pred)
    
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No Fire", "Fire"]
    )
    
    fig, ax = plt.subplots(figsize=(5,5))
    
    disp.plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format="d"
    )
    
    plt.title("Confusion Matrix")
    
    plt.tight_layout()
    
    plt.savefig(
        "confusion_matrix.png",
        dpi=600,
        bbox_inches="tight"
    )
    
    plt.show()

    target_precision = 0.2
    idx = np.where(precision >= target_precision)[0]

    if len(idx) > 0:
        chosen_t = thresholds[idx[0]]
    else:
        chosen_t = 0.5

    print("Chosen threshold:", chosen_t)

    pred_class = (pred > chosen_t).astype(int)
    print(classification_report(y, pred_class))

    # ---------- TOP-10% CLASSIFICATION REPORT ----------
    k = int(0.1 * len(pred))

    sorted_idx = np.argsort(pred)
    top_threshold = pred[sorted_idx][-k]

    print("Top-10% threshold:", top_threshold)

    pred_top = (pred > top_threshold).astype(int)

    print("Top-10% Classification Report:")
    print(classification_report(y, pred_top))

    # ---------- TOP-K (GLOBAL) ----------
    k = int(0.1 * len(pred))

    sorted_idx = np.argsort(pred)
    top_idx = sorted_idx[-k:]

    top_threshold = pred[sorted_idx][-k]

    print("Top-10% threshold:", top_threshold)
    print("Fire in top 10%:", np.mean(y[top_idx]))

    # ---------- SPATIAL ----------
    model_sp = model.__class__(**model.get_params())
    model_sp.fit(X_train, y_train)

    pred_test = model_sp.predict_proba(X_test)[:,1]

    print("Spatial AUC:", roc_auc_score(y_test, pred_test))

    # ---------- TOP-K (SPATIAL) ----------
    k_sp = int(0.1 * len(pred_test))

    sorted_idx_sp = np.argsort(pred_test)
    top_idx_sp = sorted_idx_sp[-k_sp:]

    print("Spatial Top-10% fire rate:", np.mean(y_test[top_idx_sp]))

# =========================
# 14. FULL MAP PREDICTION
# =========================
mask_full = mask

ntl2_valid = ntl2_f[mask_full]
roads_valid = roads_f[mask_full]
frag_valid = frag_f[mask_full]
ndvi_valid = ndvi_f[mask_full]

ntl2_valid = np.log1p(ntl2_valid)

ntl2_valid = normalize(ntl2_valid)
roads_valid = normalize(roads_valid)
frag_valid = normalize(frag_valid)
ndvi_valid = normalize(ndvi_valid)
ndvi_valid = 1 - ndvi_valid
lst_valid = normalize(lst_f[mask])

interaction_valid = ntl2_valid * frag_valid * (1 + roads_valid)
ntl_sq_valid = ntl2_valid ** 2
frag_sq_valid = frag_valid ** 2
risk_combo_valid = ndvi_valid * frag_valid * roads_valid

builtup_valid = normalize(builtup_f[mask_full])
aspect_valid = normalize(aspect_f[mask_full])
# optional
vpd_valid = normalize(vpd_f[mask_full])

human_pressure_valid = ntl2_valid * (1 + roads_valid) * (1 + builtup_valid)
terrain_risk_valid = aspect_valid * frag_valid

human_valid = (ntl2_valid + roads_valid + builtup_valid) / 3
fuel_valid = (ndvi_valid + frag_valid) / 2

wui_pressure_valid = human_valid * fuel_valid

X_full = np.column_stack([
    ntl2_valid,
    frag_valid,
    roads_valid,
    ndvi_valid,
    lst_valid,
    builtup_valid,
    aspect_valid,
    vpd_valid,              # ADD THIS
    interaction_valid,
    ntl_sq_valid,
    frag_sq_valid,
    risk_combo_valid,
    human_pressure_valid,
    terrain_risk_valid,
    human_valid,
    fuel_valid,
    wui_pressure_valid,
    vpd_valid * ndvi_valid  # ADD THIS
])

X_full = scaler.transform(X_full)


final_model = rf   

pred_valid = final_model.predict_proba(X_full)[:,1]

# normalize to spread values
pred_valid = (pred_valid - pred_valid.min()) / (pred_valid.max() - pred_valid.min())

p5 = np.percentile(pred_valid, 5)
p95 = np.percentile(pred_valid, 95)

pred_valid = np.clip((pred_valid - p5) / (p95 - p5), 0, 1)

rank = np.argsort(np.argsort(pred_valid))
rank = rank / len(rank)

pred_full = np.full_like(ntl2_f, np.nan, dtype=np.float32)
pred_full[mask_full] = pred_valid

pred_map = pred_full.reshape(ntl2.shape)

from scipy.ndimage import gaussian_filter

pred_map_smooth = gaussian_filter(pred_map, sigma=3)

# =========================
# 15. VISUALIZATION
# =========================
# =========================
# Reconstruct WUI-PI raster
# =========================

wui_full = np.full_like(ntl2_f, np.nan, dtype=np.float32)

wui_full[mask_full] = wui_pressure_valid

wui_map = wui_full.reshape(ntl2.shape)

from scipy.ndimage import gaussian_filter

wui_map = gaussian_filter(wui_map, sigma=2)

# Figure 6: Spatial Distribution of the WUI Pressure Index
#wui pi map
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8,7))

im = plt.imshow(
    (wui_map),
    cmap="magma",
    vmin=0,
    vmax=np.nanpercentile(wui_map,98),
    interpolation="nearest"
)

# Study area outline
plt.contour(
    ~np.isnan(wui_map),
    levels=[0.5],
    colors="black",
    linewidths=0.7
)

# Optional: observed fire locations
y_fire, x_fire = np.where(fire == 1)

plt.scatter(
    x_fire,
    y_fire,
    s=4,
    c="cyan",
    edgecolors="black",
    linewidths=0.8,
    alpha=0.35,
    label="Observed wildfire"
)

cbar = plt.colorbar(
    im,
    fraction=0.04,
    pad=0.03
)

cbar.set_label(
    "WUI Pressure Index (WUI-PI)",
    fontsize=12
)

plt.legend(
    loc="lower right",
    frameon=True,
    fontsize=10
)

plt.title(
    "WUI PI",
    fontsize=16,
    weight="bold"
)

plt.axis("off")

plt.tight_layout()

plt.savefig(
    "wui_pi_map.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()




####
error = pred_map_smooth - fire

overpred = error > 0.5
underpred = error < -0.5
correct = (~overpred) & (~underpred)

def mean_safe(arr, mask):
    return np.nanmean(arr[mask])

print("LST:", mean_safe(lst, overpred), mean_safe(lst, underpred), mean_safe(lst, correct))
print("NDVI:", mean_safe(ndvi, overpred), mean_safe(ndvi, underpred), mean_safe(ndvi, correct))
print("NTL:", mean_safe(ntl2, overpred), mean_safe(ntl2, underpred), mean_safe(ntl2, correct))
print("Roads:", mean_safe(roads, overpred), mean_safe(roads, underpred), mean_safe(roads, correct))

####

# Figure 10: Spatial Bias Maps (Overprediction / Underprediction)
from scipy.ndimage import binary_dilation
import matplotlib.pyplot as plt
import numpy as np

underpred_big = binary_dilation(underpred, iterations=2)

# Light grey background showing study area
background = np.where(np.isnan(pred_map_smooth), np.nan, 0.8)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# -------------------------
# Overprediction
# -------------------------
ax = axes[0]

ax.imshow(background, cmap="Greys", alpha=0.25)

ax.imshow(
    np.ma.masked_where(~overpred, overpred),
    cmap="Reds",
    alpha=0.95
)

ax.contour(
    ~np.isnan(pred_map_smooth),
    levels=[0.5],
    colors="black",
    linewidths=0.6
)

ax.set_title(
    "Overprediction Regions",
    fontsize=15,
    fontweight="bold"
)

ax.axis("off")

# -------------------------
# Underprediction
# -------------------------
ax = axes[1]

ax.imshow(background, cmap="Greys", alpha=0.25)

ax.imshow(
    np.ma.masked_where(~underpred_big, underpred_big),
    cmap="Blues",
    alpha=0.95
)

ax.contour(
    ~np.isnan(pred_map_smooth),
    levels=[0.5],
    colors="black",
    linewidths=0.6
)

ax.set_title(
    "Underprediction Regions",
    fontsize=15,
    fontweight="bold"
)

ax.axis("off")

plt.tight_layout()

plt.savefig(
    "spatial_bias_maps.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()






In [ ]:
# Figure 9: Feature configuration comparison (AUC vs Spatial AUC)

import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import make_interp_spline

# data
labels = ["Baseline", "+WUI", "+VPD", "+WUI+VPD"]
global_auc = [0.907, 0.911, 0.943, 0.948]
spatial_auc = [0.612, 0.614, 0.567, 0.547]

x = np.arange(len(labels))

# smooth x
x_smooth = np.linspace(x.min(), x.max(), 200)

# smooth curves
global_smooth = make_interp_spline(x, global_auc)(x_smooth)
spatial_smooth = make_interp_spline(x, spatial_auc)(x_smooth)

# plot
plt.figure(figsize=(6,6))

# smooth lines
plt.plot(x_smooth, global_smooth, linewidth=2, label="Global AUC")
plt.plot(x_smooth, spatial_smooth, linewidth=2, linestyle='--', label="Spatial AUC")

# original points
plt.scatter(x, global_auc, s=40)
plt.scatter(x, spatial_auc, s=40)

# baseline line (like ROC)
plt.axhline(y=0.5, linestyle='--', linewidth=1, alpha=0.5)

# labels
plt.xticks(x, labels)
plt.xlabel("Feature Configuration")
plt.ylabel("AUC")
plt.title("Global vs Spatial Performance")

# clean look
for spine in ["top", "right"]:
    plt.gca().spines[spine].set_visible(False)

plt.legend()

plt.tight_layout()
plt.savefig("auc_tradeoff_smooth.png", dpi=600)
plt.show()

In [ ]:
# Figure 12: Permutation feature importance

from sklearn.inspection import permutation_importance
perm = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

In [ ]:
# Figure 11: Impurity-based Random Forest feature importance

feature_names = [
    "NTL",
    "Fragmentation",
    "Roads",
    "NDVI (inv)",
    "LST",
    "Built-up",
    "Aspect",
    "VPD",
    "Interaction",
    "NTL²",
    "Frag²",
    "RiskCombo",
    "HumanPressure",
    "Terrain",
    "Human",
    "Fuel",
    "WUI-PI",
    "VPD×NDVI"
]

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# -------------------------
# Data
# -------------------------
rf_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.feature_importances_
})

rf_df = rf_df.sort_values("Importance", ascending=True)[::-1]

plt.figure(figsize=(8,7))

colors = ["#d9d9d9"] * len(rf_df)
for i in range(0,5):
    colors[i] = "#4c72b0"

bars = plt.barh(
    rf_df["Feature"],
    rf_df["Importance"],
    color=colors,
    edgecolor="black",
    linewidth=0.7
)

# Value labels
for bar, value in zip(bars, rf_df["Importance"]):
    plt.text(
        value + 0.002,
        bar.get_y() + bar.get_height()/2,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

plt.xlabel("Relative Importance")
plt.ylabel("")
plt.title("Random Forest Feature Importance")

plt.grid(axis="x", alpha=0.3)
sns.despine(left=True)

plt.tight_layout()
plt.savefig("rf_importance.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 12: Permutation feature importance

perm_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
})

# Keep the same ordering as RF
rf_order = rf_df["Feature"].tolist()

perm_df = (
    perm_df
    .set_index("Feature")
    .loc[rf_order]
    .reset_index()
)

sns.set_theme(style="whitegrid")

plt.figure(figsize=(8,7))

colors = ["#d9d9d9"] * len(perm_df)
for i in range(0,5):
    colors[i] = "#4c72b0"

bars = plt.barh(
    perm_df["Feature"],
    perm_df["Importance"],
    xerr=perm_df["Std"],
    color=colors,
    edgecolor="black",
    linewidth=0.7,
    capsize=3
)

# Value labels
for bar, value in zip(bars, perm_df["Importance"]):
    plt.text(
        value + 0.0005,
        bar.get_y() + bar.get_height()/2,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

plt.xlabel("Mean Decrease in AUC")
plt.ylabel("")
plt.title("Permutation Feature Importance")

plt.grid(axis="x", alpha=0.3)
sns.despine(left=True)

plt.tight_layout()
plt.savefig("permutation_importance.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 13: Comparison of impurity and permutation feature importance

comparison = rf_df.merge(
    perm_df[["Feature","Importance"]],
    on="Feature",
    suffixes=("_RF","_Perm")
)

# Match RF order
comparison["Feature"] = pd.Categorical(
    comparison["Feature"],
    categories=rf_df["Feature"],
    ordered=True
)

comparison = comparison.sort_values("Feature")

plt.figure(figsize=(9,8))

y = np.arange(len(comparison))
h = 0.38

plt.barh(
    y-h/2,
    comparison["Importance_RF"],
    height=h,
    color="#4c72b0",
    edgecolor="black",
    linewidth=0.6,
    label="Random Forest"
)

plt.barh(
    y+h/2,
    comparison["Importance_Perm"],
    height=h,
    color="#bfbfbf",
    edgecolor="black",
    linewidth=0.6,
    label="Permutation"
)

# Labels
for i, v in enumerate(comparison["Importance_RF"]):
    plt.text(v+0.002, i-h/2, f"{v:.3f}", va="center", fontsize=8)

for i, v in enumerate(comparison["Importance_Perm"]):
    plt.text(v+0.0005, i+h/2, f"{v:.3f}", va="center", fontsize=8)

plt.yticks(y, comparison["Feature"])

plt.xlabel("Importance")
plt.ylabel("")
plt.title("Comparison of Feature Importance Methods")

plt.grid(axis="x", alpha=0.3)
sns.despine(left=True)

plt.legend(frameon=False)

plt.tight_layout()
plt.savefig("importance_comparison.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:

# REPEATED VALIDATION
# 10 independent random background sample selections


from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    auc
)

import numpy as np
import pandas as pd



# Repeated validation seeds

seeds = [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]

results = []


# ------------------------------------------------------------
# Spatial partition
# West / left half = training
# East / right half = testing
# ------------------------------------------------------------

h, w = ntl2.shape

left_mask = np.zeros_like(ntl2, dtype=bool)
left_mask[:, :w//2] = True

right_mask = ~left_mask

left_f = left_mask.flatten()[mask]
right_f = right_mask.flatten()[mask]

X_train = X[left_f]
y_train = y[left_f]

X_test = X[right_f]
y_test = y[right_f]



# Repeated experiments


for seed in seeds:

    print("\n" + "=" * 60)
    print(f"Seed = {seed}")
    print("=" * 60)


    # Random background sampling (10:1 non-fire : fire)
   

    X_no_down = resample(
        X_no,
        n_samples=len(X_fire) * 10,
        random_state=seed
    )

    X_bal = np.vstack([
        X_fire,
        X_no_down
    ])

    y_bal = np.array(
        [1] * len(X_fire) +
        [0] * len(X_no_down)
    )


   
    # Random Forest
   

    rf_repeat = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=20,
        class_weight={0: 1, 1: 20},
        n_jobs=-1,
        random_state=42
    )

    rf_repeat.fit(
        X_bal,
        y_bal
    )


    
    # Global prediction
    

    pred = rf_repeat.predict_proba(X)[:, 1]

    global_auc = roc_auc_score(
        y,
        pred
    )


    
    # PR-AUC
    

    precision, recall, thresholds = precision_recall_curve(
        y,
        pred
    )

    pr_auc = auc(
        recall,
        precision
    )


   
    # Operating threshold
    # Minimum precision = 0.20
    

    target_precision = 0.20

    idx = np.where(
        precision >= target_precision
    )[0]

    if len(idx) > 0:
        chosen_t = thresholds[idx[0]]
    else:
        chosen_t = 0.5


    
    # Global Top-10%
    
    k = int(
        0.1 * len(pred)
    )

    sorted_idx = np.argsort(pred)

    top_idx = sorted_idx[-k:]

    top_threshold = pred[sorted_idx][-k]

    top_fire_rate = np.mean(
        y[top_idx]
    )


    
    # Spatial validation
    # West = train
    # East = test
    

    rf_spatial = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=20,
        class_weight={0: 1, 1: 20},
        n_jobs=-1,
        random_state=42
    )

    rf_spatial.fit(
        X_train,
        y_train
    )

    pred_test = rf_spatial.predict_proba(
        X_test
    )[:, 1]

    spatial_auc = roc_auc_score(
        y_test,
        pred_test
    )


    
    # Spatial Top-10%
    

    k_sp = int(
        0.1 * len(pred_test)
    )

    sorted_idx_sp = np.argsort(
        pred_test
    )

    top_idx_sp = sorted_idx_sp[-k_sp:]

    spatial_fire_rate = np.mean(
        y_test[top_idx_sp]
    )


   
    # Store results

    results.append([
        global_auc,
        pr_auc,
        chosen_t,
        top_threshold,
        top_fire_rate,
        spatial_auc,
        spatial_fire_rate
    ])



# Summary


results = np.array(results)

metric_names = [
    "Global AUC",
    "PR-AUC",
    "Operating Threshold",
    "Top-10% Threshold",
    "Top-10% Fire Rate",
    "Spatial AUC",
    "Spatial Top-10% Fire Rate"
]


summary = pd.DataFrame({
    "Metric": metric_names,
    "Mean": results.mean(axis=0),
    "Standard Deviation": results.std(axis=0)
})


print("\n")
print("=" * 70)
print("REPEATED VALIDATION SUMMARY")
print("=" * 70)

print(summary)


summary.to_csv(
    "repeated_validation_summary.csv",
    index=False
)